# Data Cleaning & Validation

## Objectives

- Standardise column names.
- Convert data types.
- Check missing values and duplicates.
- Identify invalid values.
- Validate business rules.
- Prepare data for PostgreSQL.

In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/retail_store_inventory.csv")

df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


Standardise column names

In [10]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("/", "_")
)

df.columns.tolist()

['date',
 'store_id',
 'product_id',
 'category',
 'region',
 'inventory_level',
 'units_sold',
 'units_ordered',
 'demand_forecast',
 'price',
 'discount',
 'weather_condition',
 'holiday_promotion',
 'competitor_pricing',
 'seasonality']

Convert data types

In [11]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   date                73100 non-null  datetime64[us]
 1   store_id            73100 non-null  str           
 2   product_id          73100 non-null  str           
 3   category            73100 non-null  str           
 4   region              73100 non-null  str           
 5   inventory_level     73100 non-null  int64         
 6   units_sold          73100 non-null  int64         
 7   units_ordered       73100 non-null  int64         
 8   demand_forecast     73100 non-null  float64       
 9   price               73100 non-null  float64       
 10  discount            73100 non-null  int64         
 11  weather_condition   73100 non-null  str           
 12  holiday_promotion   73100 non-null  int64         
 13  competitor_pricing  73100 non-null  float64       
 14  s

In [12]:
print("Start date:", df["date"].min())
print("End date:", df["date"].max())

Start date: 2022-01-01 00:00:00
End date: 2024-01-01 00:00:00


Missing values & duplicates

In [13]:
df.isnull().sum()

date                  0
store_id              0
product_id            0
category              0
region                0
inventory_level       0
units_sold            0
units_ordered         0
demand_forecast       0
price                 0
discount              0
weather_condition     0
holiday_promotion     0
competitor_pricing    0
seasonality           0
dtype: int64

In [14]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [15]:
df.duplicated(
    subset=["date", "store_id", "product_id"]
).sum()

np.int64(0)

Business rules validation

In [16]:
print("Negative inventory:",
      (df["inventory_level"] < 0).sum())

print("Negative units sold:",
      (df["units_sold"] < 0).sum())

print("Negative units ordered:",
      (df["units_ordered"] < 0).sum())

print("Invalid prices:",
      (df["price"] <= 0).sum())

print("Invalid discounts:",
      ((df["discount"] < 0) |
       (df["discount"] > 100)).sum())

print("Negative demand forecast:",
      (df["demand_forecast"] < 0).sum())

Negative inventory: 0
Negative units sold: 0
Negative units ordered: 0
Invalid prices: 0
Invalid discounts: 0
Negative demand forecast: 673


# Feature Engineering

## Objectives

- Calculate discounted product prices.
- Calculate sales revenue.
- Estimate inventory value.
- Identify low-stock products.
- Generate reorder recommendations.
- Extract time-based features.

In [17]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("Duplicate business keys:",
      df.duplicated(
          subset=["date", "store_id", "product_id"]
      ).sum())

Missing values: 0
Duplicate rows: 0
Duplicate business keys: 0


Discounted price

In [18]:
df["discounted_price"] = (
    df["price"] * (1 - df["discount"] / 100)
).round(2)

df[["price", "discount", "discounted_price"]].head()

,price,discount,discounted_price
0,33.50,20,26.80
1,63.01,20,50.41
2,27.99,10,25.19
3,32.72,10,29.45
4,73.64,0,73.64


Revenue

In [19]:
df["revenue"] = (
    df["units_sold"] * df["discounted_price"]
).round(2)

df[["units_sold", "discounted_price", "revenue"]].head()

,units_sold,discounted_price,revenue
0,127,26.80,3403.60
1,150,50.41,7561.50
2,65,25.19,1637.35
3,61,29.45,1796.45
4,14,73.64,1030.96


Inventory value

In [20]:
df["inventory_value"] = (
    df["inventory_level"] * df["price"]
).round(2)

df[["inventory_level", "price", "inventory_value"]].head()

,inventory_level,price,inventory_value
0,231,33.50,7738.50
1,204,63.01,12854.04
2,102,27.99,2854.98
3,469,32.72,15345.68
4,166,73.64,12224.24


Stock status

In [21]:
def get_stock_status(quantity):
    if quantity == 0:
        return "Out of Stock"
    elif quantity < 50:
        return "Low Stock"
    elif quantity < 300:
        return "In Stock"
    else:
        return "Overstock"

In [22]:
df["stock_status"] = df["inventory_level"].apply(
    get_stock_status
)

df[["inventory_level", "stock_status"]].head()

,inventory_level,stock_status
0,231,In Stock
1,204,In Stock
2,102,In Stock
3,469,Overstock
4,166,In Stock


Reorder quantity

In [23]:
df["reorder_quantity"] = np.ceil(
    df["demand_forecast"] - df["inventory_level"]
).clip(lower=0).astype(int)

In [24]:
df["reorder_required"] = df["reorder_quantity"] > 0

df[[
    "product_id",
    "inventory_level",
    "demand_forecast",
    "reorder_quantity",
    "reorder_required"
]].head()

,product_id,inventory_level,demand_forecast,reorder_quantity,reorder_required
0,P0001,231,135.47,0,False
1,P0002,204,144.04,0,False
2,P0003,102,74.02,0,False
3,P0004,469,62.18,0,False
4,P0005,166,9.26,0,False


Time-based features

In [25]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.month_name()
df["day_of_week"] = df["date"].dt.day_name()
df["quarter"] = df["date"].dt.quarter
df["is_weekend"] = df["date"].dt.dayofweek >= 5

df[[
    "date",
    "year",
    "month",
    "month_name",
    "day_of_week",
    "quarter",
    "is_weekend"
]].head()

/var/folders/_l/xdh_ryh90xz2w7x6qt5_z1nw0000gn/T/ipykernel_63217/1599679763.py:6: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["is_weekend"] = df["date"].dt.dayofweek >= 5


,date,year,month,month_name,day_of_week,quarter,is_weekend
0,2022-01-01,2022,1,January,Saturday,1,True
1,2022-01-01,2022,1,January,Saturday,1,True
2,2022-01-01,2022,1,January,Saturday,1,True
3,2022-01-01,2022,1,January,Saturday,1,True
4,2022-01-01,2022,1,January,Saturday,1,True


In [26]:
df.head()

,date,store_id,product_id,category,region,inventory_level,units_sold,units_ordered,demand_forecast,price,...,inventory_value,stock_status,reorder_quantity,reorder_required,year,month,month_name,day_of_week,quarter,is_weekend
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,...,7738.50,In Stock,0,False,2022,1,January,Saturday,1,True
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,...,12854.04,In Stock,0,False,2022,1,January,Saturday,1,True
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,...,2854.98,In Stock,0,False,2022,1,January,Saturday,1,True
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,...,15345.68,Overstock,0,False,2022,1,January,Saturday,1,True
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,...,12224.24,In Stock,0,False,2022,1,January,Saturday,1,True


In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   date                73100 non-null  datetime64[us]
 1   store_id            73100 non-null  str           
 2   product_id          73100 non-null  str           
 3   category            73100 non-null  str           
 4   region              73100 non-null  str           
 5   inventory_level     73100 non-null  int64         
 6   units_sold          73100 non-null  int64         
 7   units_ordered       73100 non-null  int64         
 8   demand_forecast     73100 non-null  float64       
 9   price               73100 non-null  float64       
 10  discount            73100 non-null  int64         
 11  weather_condition   73100 non-null  str           
 12  holiday_promotion   73100 non-null  int64         
 13  competitor_pricing  73100 non-null  float64       
 14  s

In [28]:
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])
print("Missing values:", df.isnull().sum().sum())

Total rows: 73100
Total columns: 27
Missing values: 0


In [29]:
df.to_csv(
    "../data/processed/inventory_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
